# 02 — Phase-Based Motion Magnification: Step-by-Step Walkthrough

**Phase-based magnification** is more robust than the Eulerian (intensity-based) approach because it exploits a key property of complex-valued image representations: **the local phase of a band-pass subband encodes sub-pixel spatial displacement**.

When an object moves by a small amount δ, the phase of its complex steerable pyramid subband changes by an amount proportional to δ (independent of local contrast).  Amplifying that phase change amplifies the motion — without the noise amplification that plagues intensity-based methods at high α.

**Pipeline summary:**
1. Decompose each frame into a **complex steerable pyramid** (oriented band-pass filters in the frequency domain).
2. Extract the instantaneous **phase** of each subband.
3. Compute phase deltas relative to a reference frame.
4. **Bandpass-filter** the phase-delta time-series to isolate the motion of interest.
5. Multiply filtered phase by **α** and apply it back: `subband *= exp(i·α·Δφ)`.
6. Reconstruct via IFFT and sum subbands.

> **Run this notebook from the project root directory.**
>
> Set `VIDEO_PATH` in the *Configuration* cell before running.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.io.video_io import load_video, save_video
from src.utils.phase_utils import bgr2yiq, yiq2rgb
from src.pyramids.steerable import SteerablePyramid
from src.magnification.phase_based import run_phase_based

print('Imports OK')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
VIDEO_PATH    = str(PROJECT_ROOT / 'videos' / 'wrist.avi')

FREQ_LOW      = 0.5       # Hz
FREQ_HIGH     = 2.0       # Hz
ALPHA         = 25.0      # phase amplification factor
COLORSPACE    = 'luma3'   # process Y channel; keep IQ for colour
PYRAMID_TYPE  = 'half_octave'
SIGMA         = 2.0       # amplitude-weighted Gaussian blur std-dev (0 = off)
BATCH_SIZE    = 4
SCALE         = 0.5       # spatial scale factor
OUTPUT_PATH   = str(PROJECT_ROOT / 'results' / 'videos' / 'phase_output.mp4')
# ──────────────────────────────────────────────────────────────────────────

In [ ]:
frames, fps = load_video(VIDEO_PATH, scale=SCALE)
T, H, W, C = frames.shape
print(f'{T} frames  |  {W}×{H} px  |  {fps:.2f} fps')

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(cv2.cvtColor((frames[0] * 255).astype('uint8'), cv2.COLOR_BGR2RGB))
ax.set_title('Frame 0 (original)')
ax.axis('off')
plt.tight_layout()
plt.show()

## Step 1 — Complex Steerable Pyramid Filters

The steerable pyramid tiles the 2-D frequency plane with oriented band-pass filters.  Each filter covers a specific combination of spatial scale (radial frequency band) and orientation.  Because the filters are complex-valued, each subband has both an **amplitude** (local contrast) and a **phase** (local spatial position).

In [ ]:
import numpy as np

depth = int(np.floor(np.log2(min(H, W))) - 2)
csp = SteerablePyramid(
    depth=depth, orientations=8, filters_per_octave=2,
    twidth=0.75, complex_pyr=True
)
filters, crops = csp.get_filters(H, W, cropped=False)

print(f'Pyramid depth: {depth}  |  Total filters: {len(filters)}')
print(f'  1 high-pass  +  {depth * 8 * 2} band-pass  +  1 low-pass')

# Show the magnitude of a few mid-level band-pass filters
band_start = 1                          # skip hi-pass at index 0
show_idx   = [band_start + i for i in range(min(6, len(filters) - 2))]

ncols = len(show_idx)
fig, axes = plt.subplots(1, ncols, figsize=(3 * ncols, 3))
if ncols == 1:
    axes = [axes]
for ax, idx in zip(axes, show_idx):
    ax.imshow(np.abs(filters[idx]), cmap='viridis', origin='upper')
    ax.set_title(f'Filter {idx}')
    ax.axis('off')
plt.suptitle('Steerable pyramid filter magnitudes (frequency domain)', y=1.01)
plt.tight_layout()
plt.show()

## Step 2 — Phase Extraction from a Subband

Decompose frame 0 using one of the band-pass filters and visualise the complex subband's **phase**.  Phase values near zero mean the local pattern is where it was in the reference frame; non-zero phase means the pattern has shifted.

In [ ]:
# Decompose luma of frame 0 into pyramid subbands
luma0 = bgr2yiq(frames[0])[:, :, 0].astype('float32')
pyramid0 = csp.build_pyramid(luma0, filters, crops, freq=False)

# Visualise amplitude and phase of subband index 1 (first band-pass)
subband_idx = band_start
sb = pyramid0[subband_idx]  # complex array (H, W)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(np.abs(sb), cmap='gray')
axes[0].set_title(f'Subband {subband_idx} — amplitude')
axes[0].axis('off')

axes[1].imshow(np.angle(sb), cmap='hsv', vmin=-np.pi, vmax=np.pi)
axes[1].set_title(f'Subband {subband_idx} — phase [−π, π]')
axes[1].axis('off')

plt.suptitle('Complex steerable pyramid subband (frame 0)', y=1.01)
plt.tight_layout()
plt.show()

## Step 3 — Phase-Delta Over Time (Motion Signal)

Track the phase of a single pixel across all frames in one subband.  The phase fluctuations at the pulse frequency (0.5–2 Hz) are the signal we will amplify.  This plot shows why phase-based magnification works: the sub-pixel motion creates a detectable, periodic phase change.

In [ ]:
# Compute phase at a centre pixel across all frames for subband `subband_idx`
py, px = H // 2, W // 2
ref_phase = np.angle(pyramid0[subband_idx][py, px])

phase_over_time = []
for t in range(T):
    luma_t = bgr2yiq(frames[t])[:, :, 0].astype('float32')
    pyr_t  = csp.build_pyramid(luma_t, filters, crops, freq=False)
    delta  = np.angle(pyr_t[subband_idx][py, px]) - ref_phase
    # Wrap to [-π, π]
    delta  = (np.pi + delta) % (2 * np.pi) - np.pi
    phase_over_time.append(delta)

phase_over_time = np.array(phase_over_time)
t_axis = np.arange(T) / fps

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_axis, phase_over_time, linewidth=0.9, color='darkorchid')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Phase delta (rad)')
ax.set_title(f'Phase-delta at pixel ({px},{py}), subband {subband_idx}')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Step 4 — Full Phase-Based Pipeline

`run_phase_based` chains all of the above steps — colorspace conversion, pyramid decomposition, phase filtering, amplification, and reconstruction — into a single callable.

In [ ]:
amplified = run_phase_based(
    frames=frames,
    fps=fps,
    freq_low=FREQ_LOW,
    freq_high=FREQ_HIGH,
    alpha=ALPHA,
    colorspace=COLORSPACE,
    pyramid_type=PYRAMID_TYPE,
    sigma=SIGMA,
    batch_size=BATCH_SIZE,
)

print(f'Output shape: {amplified.shape}  dtype: {amplified.dtype}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, vid, title in zip(axes, [frames, amplified], ['Original', f'Phase-based  α={ALPHA}']):
    ax.imshow(cv2.cvtColor((vid[0] * 255).astype('uint8'), cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
save_video(amplified, OUTPUT_PATH, fps=fps)
print(f'Saved → {OUTPUT_PATH}')